# Приоритет заявок после первого ответа

Самостоятельный проект по главе 7. Единица — заявка lead_id. Диспетчер оценивает её после первого ответа: уже известны исходная анкета, канал, менеджер, response_hours, повторность и запрос дегустации. Признак tasting считаю известным из заявки; перед запуском это предположение надо подтвердить владельцу CRM. До первого ответа response_hours использовать нельзя. contract_rub, converted — будущие исходы, lead_id — ключ, а не признак.

План: сначала изолировать последние 750 по времени без просмотра меток; обучать только созревшую историю; выбирать C и веса в Pipeline по временной CV с созреванием цели; зафиксировать победителя; один раз открыть итоговые метки; сравнить два списка одинакового размера и сохранить воспроизводимый Pipeline с сырыми датами. Метки доступны через created+30 дней по контракту учебной выгрузки. Неполную историю нельзя считать отрицательной.

В отличие от карточки pipe, здесь исполняю календарное ограничение вводной. Первые описательные связи и CV ниже считаются только на train; метки финала до секции «Финальная оценка» не используются.

In [1]:
from pathlib import Path
import json, joblib, numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from catering_features import LeadFeatures

ROOT = Path.cwd()
d = pd.read_csv(ROOT / 'data/catering_leads.csv')
d['created'] = pd.to_datetime(d['created'])
d = d.sort_values(['created','lead_id']).set_index('lead_id', drop=False)
final_index = d.index[-750:]
final_start = d.loc[final_index, 'created'].min()
label_ready = d['created'] + pd.Timedelta(days=30)
train_index = d.index[(d['created'] < final_start) & (label_ready < final_start)]
embargo_index = d.index[(d['created'] < final_start) & (label_ready >= final_start)]
assert not set(train_index) & set(final_index)
assert len(train_index)==1869 and len(embargo_index)==381
print({'train':len(train_index),'embargo':len(embargo_index),'final':len(final_index),'final_start':str(final_start),'train_before':str(final_start-pd.Timedelta(days=30)), 'final_labels_ready':str(label_ready.loc[final_index].max())})
raw_cols = ['created','channel','event_type','city','company_size','guests','budget_pp','days_to_event','manager','response_hours','repeat_client','tasting']
X_train=d.loc[train_index,raw_cols].copy(); y_train=d.loc[train_index,'converted'].copy()
print('Только train: конверсия',round(y_train.mean(),3))
print(d.loc[train_index].groupby('channel').converted.agg(['count','mean']).round(3))
print(X_train.isna().sum())

{'train': 1869, 'embargo': 381, 'final': 750, 'final_start': '2026-07-21 10:58:00', 'train_before': '2026-06-21 10:58:00', 'final_labels_ready': '2026-10-25 21:48:00'}
Только train: конверсия 0.134
              count   mean
channel                   
выставка        114  0.114
звонок          375  0.208
рекомендация    234  0.359
сайт            722  0.089
соцсети         355  0.025
тендер           69  0.043
created             0
channel             0
event_type          0
city                0
company_size      160
guests              0
budget_pp         478
days_to_event       0
manager             0
response_hours      0
repeat_client       0
tasting             0
dtype: int64


Каждый CV-фолд имитирует следующий календарный период. Обучающие метки должны созреть до первой даты проверочной части. Не перемешиваю будущее с прошлым; обучаемые медианы, масштабы, категории целиком внутри Pipeline. Эти фолды не независимы, их std не является готовым доверительным интервалом. Выбор ниже — по среднему ROC-AUC; бизнес-проверка использует фиксированное число мест.

In [2]:
num=['guests','budget_pp','days_to_event','response_hours','repeat_client','tasting','hour','weekday','evening']
cat=['channel','event_type','city','company_size','manager']
pre=ColumnTransformer([
 ('num',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),num),
 ('cat',make_pipeline(SimpleImputer(strategy='constant',fill_value='не указан'),OneHotEncoder(handle_unknown='ignore')),cat)])
pipe=Pipeline([('features',LeadFeatures()),('prep',pre),('model',LogisticRegression(max_iter=2000))])
n=len(X_train); points=[n//2,2*n//3,5*n//6,n];cv=[]
for start,end in zip(points[:-1],points[1:]):
 val=np.arange(start,end); boundary=X_train.iloc[start].created
 fit=np.where((X_train.created+pd.Timedelta(days=30)<boundary).to_numpy())[0]
 assert len(fit)>0 and set(y_train.iloc[fit])=={0,1} and set(y_train.iloc[val])=={0,1}
 cv.append((fit,val))
 print('CV',len(fit),len(val),'validation starts',boundary)
search=GridSearchCV(pipe,{'model__C':[.01,.1,1,10],'model__class_weight':[None,'balanced']},cv=cv,scoring='roc_auc',error_score='raise').fit(X_train,y_train)
cv_table=pd.DataFrame(search.cv_results_).sort_values('rank_test_score')
print(cv_table[['params','mean_test_score','std_test_score']].to_string(index=False))
chosen=search.best_estimator_
print('Зафиксирован выбор:',search.best_params_, 'CV-AUC',round(search.best_score_,3))
cv_table.to_csv(ROOT/'cv_results.csv',index=False)

CV 608 312 validation starts 2026-04-01 11:05:00
CV 915 311 validation starts 2026-04-28 19:46:00
CV 1215 312 validation starts 2026-05-26 10:36:00


                                               params  mean_test_score  std_test_score
         {'model__C': 1, 'model__class_weight': None}         0.852812        0.039460
   {'model__C': 1, 'model__class_weight': 'balanced'}         0.851948        0.032123
 {'model__C': 0.1, 'model__class_weight': 'balanced'}         0.850845        0.026588
       {'model__C': 0.1, 'model__class_weight': None}         0.849750        0.025430
        {'model__C': 10, 'model__class_weight': None}         0.848316        0.036639
  {'model__C': 10, 'model__class_weight': 'balanced'}         0.846584        0.027524
{'model__C': 0.01, 'model__class_weight': 'balanced'}         0.835744        0.013612
      {'model__C': 0.01, 'model__class_weight': None}         0.828979        0.010975
Зафиксирован выбор: {'model__C': 1, 'model__class_weight': None} CV-AUC 0.853


## Финальная оценка — одно открытие

Теперь настройки зафиксированы. Обе очереди относятся к одним 750 заявкам и одному моменту принятия решения. Правило: повторный клиент или рекомендация. Для честного сравнения модель получает ровно столько мест, сколько заняло правило. Ничьи риска разрешаю по lead_id по возрастанию; сохраняю ID, а не только округлённую долю. Отдельно показываю топ-30% на 225 мест. Исторические метрики описывают ранжирование при прежнем процессе работы, не причинный эффект усиленного сопровождения.

In [3]:
X_final=d.loc[final_index,raw_cols].copy()
y_final=d.loc[final_index,'converted'].copy()
scores=pd.Series(chosen.predict_proba(X_final)[:,1],index=final_index,name='score')
rule=(X_final.repeat_client==1)|(X_final.channel=='рекомендация')
k=int(rule.sum())
rank=pd.DataFrame({'lead_id':final_index,'score':scores.to_numpy()}).sort_values(['score','lead_id'],ascending=[False,True])
model_ids=rank.head(k).lead_id.tolist()
rule_ids=sorted(X_final.index[rule].tolist())
assert len(model_ids)==len(rule_ids)==k and len(set(model_ids))==k

def queue_metrics(ids):
 found=int(y_final.loc[ids].sum())
 return {'places':len(ids),'contracts':found,'precision':found/len(ids),'recall':found/int(y_final.sum())}
report={'roc_auc':float(roc_auc_score(y_final,scores)),
 'f1_at_05':float(f1_score(y_final,chosen.predict(X_final))),
 'final_contracts':int(y_final.sum()),'model_equal_budget':queue_metrics(model_ids),
 'rule_equal_budget':queue_metrics(rule_ids),'model_top30':queue_metrics(rank.head(int(len(rank)*.3)).lead_id.tolist())}
print(json.dumps(report,ensure_ascii=False,indent=2))
pd.DataFrame({'lead_id':model_ids,'position':range(1,k+1)}).to_csv(ROOT/'model_queue.csv',index=False)
pd.DataFrame({'lead_id':rule_ids,'position':range(1,k+1)}).to_csv(ROOT/'rule_queue.csv',index=False)
rank.to_csv(ROOT/'final_ranking.csv',index=False)
(ROOT/'metrics.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
for name,ix in [('train',train_index),('embargo',embargo_index),('final',final_index)]:pd.Series(ix,name='lead_id').to_csv(ROOT/(name+'_ids.csv'),index=False)
print('Последний допустимый контакт модели:',model_ids[-1], '; следующий:',rank.iloc[k].lead_id)
print('Число равных граничной оценке:',int((rank.score==rank.iloc[k-1].score).sum()))

{
  "roc_auc": 0.8575019460300987,
  "f1_at_05": 0.39416058394160586,
  "final_contracts": 94,
  "model_equal_budget": {
    "places": 166,
    "contracts": 64,
    "precision": 0.3855421686746988,
    "recall": 0.6808510638297872
  },
  "rule_equal_budget": {
    "places": 166,
    "contracts": 57,
    "precision": 0.3433734939759036,
    "recall": 0.6063829787234043
  },
  "model_top30": {
    "places": 225,
    "contracts": 72,
    "precision": 0.32,
    "recall": 0.7659574468085106
  }
}
Последний допустимый контакт модели: L-22537 ; следующий: L-22487
Число равных граничной оценке: 1


## Передача и новая сырая заявка

Сохраняю модель, которая обучена только на 1869 созревших исторических заявках; именно к ней относятся итоговые метрики. Автоматического refit на финале нет. Файл catering_features.py нужен рядом с joblib, чтобы другой процесс мог импортировать класс. С ним передаются данные, CV-таблица, метрики и списки ID. Новая дата преобразуется внутри Pipeline, неизвестная категория кодируется нулями известных флагов. Такие категории надо мониторить.

In [4]:
joblib.dump(chosen,ROOT/'catering_model.joblib')
restored=joblib.load(ROOT/'catering_model.joblib')
assert np.array_equal(restored.predict(X_final),chosen.predict(X_final))
new=X_train.iloc[[0]].copy()
new.index=pd.Index(['NEW-001'],name='lead_id')
new['created']='2026-11-16 20:15'
new['city']='Новый неизвестный город'
new['budget_pp']=np.nan
new_score=restored.predict_proba(new)[:,1]
assert np.isfinite(new_score).all()
print('Новая сырая дата/незнакомый город, риск договора:',new_score.tolist())
new.to_csv(ROOT/'new_raw.csv',index=True)
metadata={'prediction_moment':'after first response','label_ready':'created + 30 days','train_rows':len(X_train),'final_rows':len(X_final),'final_start':str(final_start),'parameters':search.best_params_,'model_file':'catering_model.joblib','required_module':'catering_features.py','tie_rule':'score descending, lead_id ascending'}
(ROOT/'model_passport.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))

Новая сырая дата/незнакомый город, риск договора: [0.20470522045647888]


383

Сравнение показывает, сколько существующих договоров модель концентрирует в доступной очереди. Доказательства того, что звонок создаёт дополнительный договор, здесь нет. Следующий шаг бизнеса — контролируемый пилот при одинаковом числе мест с заранее выбранной метрикой дополнительной маржи; менять C по финалу нельзя. Качество на новой ноябрьской заявке неизвестно, её вероятность — прогноз, а не отдельная проверенная метрика.

## Уточнения после исправленного учебного текста

В CSV — синтетическая CRM. converted означает договор в течение 30 дней от created; фактическое время подписи отсутствует. Поля анкеты, назначенный менеджер и tasting в учебном сценарии известны при регистрации, response_hours — после первого ответа. В реальной CRM эти моменты проверяют по журналу изменений. Без времени подписи нельзя исключить уже подписанные договоры из очереди на момент первого ответа; поэтому этот результат — учебная ретроспектива, а не готовая эксплуатационная оценка. Последняя финальная метка созревает 25 октября в21:48.

Сохраняю исходный честный дизайн из трёх временных CV-фолдов с созреванием цели. Учебный KFold внутри уже зрелого train отвечает иному способу выбора фиксированного выпуска. Изменять свой выбор после просмотра финала было бы подгонкой. Повторный чистый запуск ниже воспроизводит ту же зафиксированную процедуру; новых настроек по финалу не выбираю. Общий AUC не является гарантией качества заданных166/225мест.

In [5]:
import sys, sklearn
versions={'python':sys.version.split()[0],'scikit-learn':sklearn.__version__,'pandas':pd.__version__,'numpy':np.__version__,'joblib':joblib.__version__}
(ROOT/'versions.json').write_text(json.dumps(versions,indent=2))
metadata.update({'data_kind':'synthetic CRM','target_definition':'contract within 30 days after created','signature_time_available':False,'registration_features':['questionnaire','manager','tasting'],'response_hours_available':'first response','cv_design':'3 expanding temporal folds, labels mature before validation start','final_label_maturity':'2026-10-25 21:48','operational_limit':'cannot exclude already signed contracts without signature timestamps','versions':versions})
(ROOT/'model_passport.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))
print(versions)

{'python': '3.12.14', 'scikit-learn': '1.8.0', 'pandas': '3.0.6', 'numpy': '2.4.6', 'joblib': '1.6.0'}
